# 10 — Qwen3.5-2B Full Retraining after the Five Deletion Requests

**Purpose:** create the exact Full Retraining references needed for the Qwen machine-unlearning extension.

This notebook reuses the same frozen kidney-transplant dataset, permanent split, 18 classifier features, and five deletion-scenario memberships already established by the MLP study.

For each scenario, a **fresh** `Qwen3.5-2B-Base` model is loaded. A new two-class head and new LoRA adapters are trained using **retained training rows only**. The trained Qwen baseline is used only to recover the frozen experiment configuration and threshold; it is never used to initialise Full Retraining.

Full Retraining therefore represents the reference outcome in which the forgotten training records had never been used for the task-specific Qwen adaptation.

> This retrains the kidney-task LoRA adaptation and classification head. It does not retrain Qwen's original pretrained base model.

## Pipeline

1. Verify RunPod CUDA and the repository.
2. Load the frozen dataset, feature contract and permanent split.
3. Load Notebook 02's saved deletion membership.
4. Reconstruct and verify the five exact forget/retain sets: **426, 1,992, 4,314, 4,148, 6,262** forgotten training rows.
5. Load the final Qwen **1e-5** baseline configuration only to freeze the training contract and threshold.
6. Use the same plain deterministic feature-to-text serialisation.
7. For every scenario, load fresh pretrained Qwen3.5-2B-Base + fresh LoRA + fresh classification head and train only on retained training rows.
8. Select the best checkpoint by retained-validation PR-AUC.
9. Evaluate retained-test utility and save forget-set probabilities.
10. Save each scenario immediately so completed work survives a later kernel interruption.

## 1. RunPod environment and repository

In [1]:
from pathlib import Path
import gc, json, os, random, time
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from IPython.display import display

REPO_ROOT = Path('/workspace/qub-machine-unlearning')
QWEN_ROOT = Path('/workspace/qwen35_classifier_runpod')

if not REPO_ROOT.exists():
    raise FileNotFoundError(f'Repository not found: {REPO_ROOT}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable. Reconnect to the A100 RunPod kernel before continuing.')

DEVICE = torch.device('cuda')
print('Repository:', REPO_ROOT)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

Repository: /workspace/qub-machine-unlearning
PyTorch: 2.8.0+cu128
GPU: NVIDIA A100-SXM4-80GB


In [2]:
# Import Unsloth before Transformers so its patches are applied.
import unsloth
from unsloth import FastVisionModel
from transformers import AutoProcessor
from peft import PeftModel
from sklearn.metrics import (
    average_precision_score, balanced_accuracy_score, confusion_matrix,
    f1_score, log_loss, precision_score, recall_score, roc_auc_score,
)
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
print('Qwen/Unsloth imports: OK')

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Qwen/Unsloth imports: OK


## 2. Reproducibility and output folders

In [3]:
SEED = 42

def set_reproducible_seed(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_reproducible_seed()
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_ROOT = QWEN_ROOT / 'full_retraining' / RUN_ID
MODEL_ROOT = RUN_ROOT / 'models'
RESULT_ROOT = RUN_ROOT / 'results'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print('Run ID:', RUN_ID)
print('Output root:', RUN_ROOT)

Run ID: 20260829T185449Z
Output root: /workspace/qwen35_classifier_runpod/full_retraining/20260829T185449Z


## 3. Locate the frozen project inputs

In [4]:
def locate_submission_root(repo_root):
    candidates = [
        repo_root / 'code' / 'final_submission',
        repo_root / 'final_submission',
        repo_root,
    ]

    for candidate in candidates:
        if (candidate / 'data/final/kidney_transplant_assessments.csv').is_file():
            return candidate

    raise FileNotFoundError(
        'Could not locate the final submission dataset.'
    )
ROOT = locate_submission_root(REPO_ROOT)
DATA_DIR = ROOT / 'data/final'
PROCESSED_DIR = ROOT / 'processed_data'
ASSESSMENT_PATH = DATA_DIR / 'kidney_transplant_assessments.csv'
FEATURE_PATH = DATA_DIR / 'classifier_feature_list.json'
DELETION_AUDIT_PATH = DATA_DIR / 'deletion_scenario_audit.csv'
SPLIT_PATH = PROCESSED_DIR / 'split_assignments.csv'
MEMBERSHIP_PATH = PROCESSED_DIR / 'deletion_scenario_membership.csv'

required = [ASSESSMENT_PATH, FEATURE_PATH, DELETION_AUDIT_PATH, SPLIT_PATH, MEMBERSHIP_PATH]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError('Missing required frozen inputs: ' + '; '.join(missing))

display(pd.Series({
    'assessments': str(ASSESSMENT_PATH),
    'feature contract': str(FEATURE_PATH),
    'deletion audit': str(DELETION_AUDIT_PATH),
    'split assignments': str(SPLIT_PATH),
    'deletion membership': str(MEMBERSHIP_PATH),
}, name='path').to_frame())

,path
assessments,/workspace/qub-machine-unlearning/code/final_s...
feature contract,/workspace/qub-machine-unlearning/code/final_s...
deletion audit,/workspace/qub-machine-unlearning/code/final_s...
split assignments,/workspace/qub-machine-unlearning/code/final_s...
deletion membership,/workspace/qub-machine-unlearning/code/final_s...


## 4. Choose the final Qwen baseline run

Set `BASELINE_RUN_ID` to the run ID for the **final Qwen 1e-5 baseline**. This notebook deliberately does not guess which historical run is final.

If the placeholder remains, the cell lists available run IDs and stops before any expensive training.

In [6]:
BASELINE_RUN_ID = '20260829T151430Z'

BASELINE_RESULT_DIR = QWEN_ROOT / 'results' / BASELINE_RUN_ID
BASELINE_CONFIG_PATH = BASELINE_RESULT_DIR / 'experiment_configuration.json'
BASELINE_THRESHOLD_PATH = BASELINE_RESULT_DIR / 'selected_threshold.json'
BASELINE_SERIALISATION_PATH = BASELINE_RESULT_DIR / 'serialisation_specification.json'

if BASELINE_RUN_ID.startswith('REPLACE_') or not BASELINE_CONFIG_PATH.is_file():
    print('Available Qwen result run IDs:')
    available = sorted([p.name for p in (QWEN_ROOT / 'results').glob('*') if p.is_dir()])
    for run in available[-20:]:
        print('  ', run)
    raise RuntimeError('Set BASELINE_RUN_ID above to the final Qwen 1e-5 run.')

baseline_config = json.loads(BASELINE_CONFIG_PATH.read_text(encoding='utf-8'))
baseline_threshold = json.loads(BASELINE_THRESHOLD_PATH.read_text(encoding='utf-8'))
baseline_serialisation = json.loads(BASELINE_SERIALISATION_PATH.read_text(encoding='utf-8'))
MODEL_ID = baseline_config['model_id']
FROZEN_THRESHOLD = float(baseline_threshold['threshold'])

display(pd.Series(baseline_config['training'], name='value').to_frame())
print('Baseline run:', BASELINE_RUN_ID)
print('Frozen threshold:', FROZEN_THRESHOLD)
assert MODEL_ID == 'unsloth/Qwen3.5-2B-Base'
assert float(baseline_config['training']['learning rate']) == 1e-5
assert int(baseline_config['training']['LoRA rank']) == 16
assert int(baseline_config['training']['LoRA alpha']) == 16
assert float(baseline_config['training']['LoRA dropout']) == 0

,value
model,unsloth/Qwen3.5-2B-Base
batch size,8
effective batch size,32
learning rate,0.00001
maximum epochs,10
LoRA rank,16
LoRA alpha,16
LoRA dropout,0
LoRA targets,all linear language layers (hybrid Qwen3.5 cov...
maximum sequence length,216


Baseline run: 20260829T151430Z
Frozen threshold: 0.55


## 5. Load the frozen dataset, feature contract and permanent split

In [7]:
assessments = pd.read_csv(ASSESSMENT_PATH)
feature_contract = json.loads(FEATURE_PATH.read_text(encoding='utf-8'))
split_assignments = pd.read_csv(SPLIT_PATH)
TARGET = feature_contract['target']
FEATURES = feature_contract['classifier_features']
EXPECTED_FEATURES = [
    'recipient_age', 'donor_age', 'donor_type', 'kidney_failure_cause',
    'previous_transplant', 'dialysis_months', 'abo_compatibility_category',
    'hla_mismatch_count', 'antibody_risk_score', 'cold_ischaemia_hours',
    'days_since_transplant', 'creatinine_mg_dl', 'creatinine_change_pct',
    'urine_output_ml_24h', 'tacrolimus_level_ng_ml',
    'medication_adherence_pct', 'infection_indicator', 'previous_rejection',
]
BLOCKED_COLUMNS = {
    'assessment_id', 'recipient_id', 'donor_id', 'hospital_id', 'assessment_date',
    'training_consent_status', 'training_consent_version', 'retention_expiry_date', TARGET,
}
assert len(assessments) == 60_000
assert TARGET == 'acute_rejection_within_30_days'
assert FEATURES == EXPECTED_FEATURES
assert set(FEATURES).isdisjoint(BLOCKED_COLUMNS)
assert split_assignments['recipient_id'].is_unique

data = assessments.merge(
    split_assignments[['recipient_id', 'donor_id', 'split']],
    on=['recipient_id', 'donor_id'], how='left', validate='many_to_one'
)
assert data['split'].notna().all()
split_frames = {name: data.loc[data['split'].eq(name)].copy().reset_index(drop=True)
                for name in ['train', 'validation', 'test']}
assert {k: len(v) for k,v in split_frames.items()} == {'train': 42024, 'validation': 8988, 'test': 8988}
for frame in split_frames.values():
    frame['label'] = frame[TARGET].astype('int64')

display(pd.DataFrame([
    {'split': name, 'rows': len(frame), 'positives': int(frame[TARGET].sum()), 'prevalence': frame[TARGET].mean()}
    for name, frame in split_frames.items()
]))

,split,rows,positives,prevalence
0,train,42024,3160,0.075195
1,validation,8988,669,0.074433
2,test,8988,665,0.073988


## 6. Reuse the exact five deletion memberships from Notebook 02

In [8]:
SCENARIOS = ['recipient_withdrawal','donor_withdrawal','hospital_removal','invalid_consent','retention_expiry']
EXPECTED_TRAINING_FORGET_COUNTS = {
    'recipient_withdrawal': 426,
    'donor_withdrawal': 1992,
    'hospital_removal': 4314,
    'invalid_consent': 4148,
    'retention_expiry': 6262,
}
membership = pd.read_csv(MEMBERSHIP_PATH)
delection_audit = pd.read_csv(DELETION_AUDIT_PATH).set_index('scenario')
assert set(membership['scenario']) == set(SCENARIOS)
assert not membership.duplicated(['scenario','assessment_id']).any()
assert list(delection_audit.index) == SCENARIOS

scenario_sets = {}
for scenario in SCENARIOS:
    sm = membership.loc[membership['scenario'].eq(scenario)]
    forget_ids = set(sm.loc[sm['membership_type'].eq('training_forget'),'assessment_id'])
    deleted_validation_ids = set(sm.loc[sm['membership_type'].eq('deleted_validation'),'assessment_id'])
    deleted_test_ids = set(sm.loc[sm['membership_type'].eq('deleted_test'),'assessment_id'])

    training_forget = split_frames['train'].loc[split_frames['train']['assessment_id'].isin(forget_ids)].copy()
    retained_train = split_frames['train'].loc[~split_frames['train']['assessment_id'].isin(forget_ids)].copy()
    retained_validation = split_frames['validation'].loc[~split_frames['validation']['assessment_id'].isin(deleted_validation_ids)].copy()
    retained_test = split_frames['test'].loc[~split_frames['test']['assessment_id'].isin(deleted_test_ids)].copy()

    assert len(training_forget) == EXPECTED_TRAINING_FORGET_COUNTS[scenario]
    assert len(training_forget) + len(retained_train) == 42024
    assert set(training_forget['assessment_id']).isdisjoint(set(retained_train['assessment_id']))

    scenario_sets[scenario] = {
        'training_forget': training_forget,
        'retained_train': retained_train,
        'retained_validation': retained_validation,
        'retained_test': retained_test,
    }

scenario_size_table = pd.DataFrame([
    {'scenario': s,
     'training_forget_rows': len(p['training_forget']),
     'retained_training_rows': len(p['retained_train']),
     'retained_validation_rows': len(p['retained_validation']),
     'retained_test_rows': len(p['retained_test'])}
    for s,p in scenario_sets.items()
])
display(scenario_size_table)
assert scenario_size_table['training_forget_rows'].tolist() == [426,1992,4314,4148,6262]
print('Deletion membership verification: PASS')

,scenario,training_forget_rows,retained_training_rows,retained_validation_rows,retained_test_rows
0,recipient_withdrawal,426,41598,8904,8898
1,donor_withdrawal,1992,40032,8472,8496
2,hospital_removal,4314,37710,8166,8124
3,invalid_consent,4148,37876,8046,8078
4,retention_expiry,6262,35762,7624,7614


Deletion membership verification: PASS


## 7. Same plain deterministic Qwen serialisation

In [9]:
FEATURE_LABELS = baseline_serialisation['display_labels']
BINARY_FEATURES = {'previous_transplant','infection_indicator','previous_rejection'}
SERIALISATION_VERSION = baseline_config['serialisation_version']
assert baseline_serialisation['feature_order'] == FEATURES
assert baseline_serialisation['target_included'] is False
assert baseline_serialisation['identifiers_included'] is False

def format_feature_value(feature, value):
    if pd.isna(value): return 'missing'
    if feature in BINARY_FEATURES: return 'yes' if int(value) == 1 else 'no'
    if isinstance(value, (float, np.floating)):
        return f'{float(value):.4f}'.rstrip('0').rstrip('.')
    return str(value).strip()

def serialize_assessment(row):
    return '\n'.join(
        f'{FEATURE_LABELS[feature]}: {format_feature_value(feature, row[feature])}.'
        for feature in FEATURES
    )

text_by_id = data.assign(text=data.apply(serialize_assessment, axis=1)).set_index('assessment_id')['text']
for parts in scenario_sets.values():
    for frame in parts.values():
        frame['text'] = frame['assessment_id'].map(text_by_id)
        frame['label'] = frame[TARGET].astype('int64')
        assert frame['text'].notna().all()

print('Serialisation version:', SERIALISATION_VERSION)
print(scenario_sets['recipient_withdrawal']['retained_train']['text'].iloc[0])

Serialisation version: kidney-qwen35-v1
Recipient age (years): 59.
Donor age (years): 46.
Donor type: Deceased.
Kidney failure cause: Polycystic kidney disease.
Previous transplant: no.
Dialysis duration (months): 4.
ABO compatibility: Managed incompatibility.
HLA mismatch count: 5.
Antibody risk score: 0.884.
Cold ischaemia time (hours): 4.62.
Days since transplant: 7.
Creatinine (mg/dL): 1.24.
Creatinine change (percent): -3.6519.
Urine output (mL/24h): 2450.6.
Tacrolimus level (ng/mL): 9.88.
Medication adherence (percent): 86.3.
Infection indicator: no.
Previous rejection: no.


## 8. Freeze the final Qwen training contract

In [11]:
MAX_SEQ_LENGTH = int(baseline_config['max_seq_length'])
BATCH_SIZE = int(baseline_config['training']['batch size'])
LEARNING_RATE = float(baseline_config['training']['learning rate'])
MAX_EPOCHS = int(baseline_config['training']['maximum epochs'])
WEIGHT_DECAY = float(baseline_config['training']['weight decay'])
EARLY_STOPPING_PATIENCE = int(baseline_config['training']['early stopping patience'])
GRADIENT_ACCUMULATION_STEPS = int(baseline_config['training']['effective batch size'] // BATCH_SIZE)
GRADIENT_CLIP_NORM = 1.0

print("MAX_EPOCHS from final baseline:", MAX_EPOCHS)
print("Full saved training config:")
display(pd.Series(baseline_config['training']))

assert BATCH_SIZE == 8
assert GRADIENT_ACCUMULATION_STEPS == 4
assert LEARNING_RATE == 1e-5
assert int(baseline_config['training']['LoRA rank']) == 16

display(pd.Series({
    'model': MODEL_ID, 'batch size': BATCH_SIZE,
    'gradient accumulation': GRADIENT_ACCUMULATION_STEPS,
    'effective batch size': BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS,
    'learning rate': LEARNING_RATE, 'max epochs': MAX_EPOCHS,
    'weight decay': WEIGHT_DECAY, 'early stopping patience': EARLY_STOPPING_PATIENCE,
    'LoRA rank': 16, 'LoRA alpha': 16, 'LoRA dropout': 0,
    'LoRA targets': 'all-linear', 'max sequence length': MAX_SEQ_LENGTH,
    'frozen threshold': FROZEN_THRESHOLD,
}, name='value').to_frame())

MAX_EPOCHS from final baseline: 10
Full saved training config:


model                                                unsloth/Qwen3.5-2B-Base
batch size                                                                 8
effective batch size                                                      32
learning rate                                                        0.00001
maximum epochs                                                            10
LoRA rank                                                                 16
LoRA alpha                                                                16
LoRA dropout                                                               0
LoRA targets               all linear language layers (hybrid Qwen3.5 cov...
maximum sequence length                                                  216
optimiser                                                              AdamW
weight decay                                                            0.01
early stopping patience                                                    2

,value
model,unsloth/Qwen3.5-2B-Base
batch size,8
gradient accumulation,4
effective batch size,32
learning rate,0.00001
max epochs,10
weight decay,0.01
early stopping patience,2
LoRA rank,16
LoRA alpha,16


## 9. Dataset, batching and evaluation helpers

In [12]:
class AssessmentTextDataset(Dataset):
    def __init__(self, frame):
        self.texts = frame['text'].tolist()
        self.labels = frame['label'].astype(int).tolist()
        self.ids = frame['assessment_id'].astype(str).tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i): return {'text':self.texts[i],'label':self.labels[i],'assessment_id':self.ids[i]}

def make_collate(tokenizer):
    def collate(rows):
        encoded = tokenizer([r['text'] for r in rows], padding=True, truncation=True,
                            max_length=MAX_SEQ_LENGTH, return_tensors='pt')
        return {'input_ids':encoded['input_ids'],'attention_mask':encoded['attention_mask'],
                'labels':torch.tensor([r['label'] for r in rows],dtype=torch.long),
                'assessment_id':[r['assessment_id'] for r in rows]}
    return collate

def final_token_logits(model_object, batch):
    input_ids = batch['input_ids'].to(DEVICE)
    attention_mask = batch['attention_mask'].to(DEVICE)
    sequence_logits = model_object(input_ids=input_ids, attention_mask=attention_mask).logits
    final_indices = attention_mask.sum(dim=1) - 1
    logits = sequence_logits[torch.arange(input_ids.shape[0],device=DEVICE), final_indices]
    if logits.shape != (input_ids.shape[0],2): raise RuntimeError(f'Unexpected logits shape {tuple(logits.shape)}')
    return logits

@torch.no_grad()
def evaluate_classifier(model_object, loader):
    model_object.eval(); losses=[]; labels=[]; probs=[]; ids=[]
    for batch in loader:
        y=batch['labels'].to(DEVICE); logits=final_token_logits(model_object,batch)
        loss=F.cross_entropy(logits.float(),y); p=torch.softmax(logits.float(),dim=1)[:,1]
        losses.append(float(loss.item())*len(y)); labels.extend(y.cpu().numpy()); probs.extend(p.cpu().numpy()); ids.extend(batch['assessment_id'])
    labels=np.asarray(labels,dtype=np.int64); probs=np.asarray(probs,dtype=np.float64)
    return {'loss':sum(losses)/len(labels),'pr_auc':average_precision_score(labels,probs),
            'labels':labels,'probabilities':probs,'assessment_ids':ids}

def compute_metrics(labels, probabilities, threshold):
    pred=(probabilities>=threshold).astype(np.int64)
    tn,fp,fn,tp=confusion_matrix(labels,pred,labels=[0,1]).ravel()
    return {'n':len(labels),'positive_count':int(labels.sum()),'prevalence':float(labels.mean()),
            'threshold':float(threshold),'pr_auc':float(average_precision_score(labels,probabilities)),
            'balanced_accuracy':float(balanced_accuracy_score(labels,pred)),
            'binary_cross_entropy':float(log_loss(labels,probabilities,labels=[0,1])),
            'f1':float(f1_score(labels,pred,zero_division=0)),'auroc':float(roc_auc_score(labels,probabilities)),
            'precision':float(precision_score(labels,pred,zero_division=0)),
            'recall':float(recall_score(labels,pred,zero_division=0)),
            'specificity':float(tn/(tn+fp)),'true_negative':int(tn),'false_positive':int(fp),'false_negative':int(fn),'true_positive':int(tp)}

## 10. Build a fresh Qwen classifier for each scenario

This function never loads the kidney-trained baseline adapter. Every Full Retraining model begins from the same pretrained Qwen base plus a newly initialised two-class head and new LoRA parameters.

In [13]:
class FP32ClassificationHead(nn.Linear):
    def forward(self, hidden_states):
        return F.linear(hidden_states.float(), self.weight, self.bias)

def build_fresh_classifier():
    set_reproducible_seed(SEED)
    torch.cuda.empty_cache()
    model, processor = FastVisionModel.from_pretrained(
        MODEL_ID, load_in_4bit=False, load_in_16bit=True,
        max_seq_length=MAX_SEQ_LENGTH, use_gradient_checkpointing=False,
    )
    tokenizer = getattr(processor,'tokenizer',processor)
    tokenizer.padding_side='right'
    if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
    old_head=model.get_output_embeddings()
    new_head=FP32ClassificationHead(old_head.in_features,2,bias=False,
                                    device=old_head.weight.device,dtype=torch.float32)
    nn.init.normal_(new_head.weight,mean=0.0,std=0.02)
    model.set_output_embeddings(new_head)
    model.config.num_labels=2; model.config.pad_token_id=tokenizer.pad_token_id
    model=FastVisionModel.get_peft_model(
        model, finetune_vision_layers=False, finetune_language_layers=True,
        finetune_attention_modules=True, finetune_mlp_modules=True,
        target_modules='all-linear', modules_to_save=['lm_head'],
        r=16, lora_alpha=16, lora_dropout=0, bias='none',
        use_gradient_checkpointing=False, random_state=SEED,
        use_rslora=False, loftq_config=None,
    )
    model.config.use_cache=False
    for p in model.parameters():
        if p.requires_grad: p.data=p.data.float()
    assert not [n for n,p in model.named_parameters() if p.requires_grad and p.dtype==torch.float16]
    return model, processor, tokenizer

## 11. Train one Full Retraining reference

Class weights are recomputed from each retained training set only. Validation PR-AUC selects the checkpoint. The **original final Qwen threshold is frozen across scenarios**, so threshold-dependent utility remains directly comparable.

In [14]:
def train_full_retraining_scenario(scenario, parts):
    set_reproducible_seed(SEED)
    model_dir=MODEL_ROOT/scenario; result_dir=RESULT_ROOT/scenario; best_dir=model_dir/'best_adapter'
    model_dir.mkdir(parents=True,exist_ok=True); result_dir.mkdir(parents=True,exist_ok=True)
    complete=result_dir/'COMPLETE.json'
    if complete.is_file():
        print(scenario, 'already complete — skipping')
        return json.loads(complete.read_text())

    train_frame=parts['retained_train'].reset_index(drop=True)
    val_frame=parts['retained_validation'].reset_index(drop=True)
    test_frame=parts['retained_test'].reset_index(drop=True)
    forget_frame=parts['training_forget'].reset_index(drop=True)
    assert set(train_frame['assessment_id']).isdisjoint(set(forget_frame['assessment_id']))

    model, processor, tokenizer=build_fresh_classifier(); collate=make_collate(tokenizer)
    gen=torch.Generator().manual_seed(SEED)
    train_loader=DataLoader(AssessmentTextDataset(train_frame),batch_size=BATCH_SIZE,shuffle=True,generator=gen,collate_fn=collate,num_workers=0)
    val_loader=DataLoader(AssessmentTextDataset(val_frame),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate,num_workers=0)

    weights=compute_class_weight(class_weight='balanced',classes=np.array([0,1]),y=train_frame['label'].to_numpy())
    class_weights=torch.tensor(weights,dtype=torch.float32,device=DEVICE)
    trainable=[p for p in model.parameters() if p.requires_grad]
    optimiser=torch.optim.AdamW(trainable,lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
    steps_per_epoch=int(np.ceil(len(train_loader)/GRADIENT_ACCUMULATION_STEPS))
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimiser,T_max=max(steps_per_epoch*MAX_EPOCHS,1))

    best_pr=-np.inf; best_loss=np.inf; best_epoch=None; no_improve=0; history=[]
    started=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    for epoch in range(1,MAX_EPOCHS+1):
        model.train(); optimiser.zero_grad(set_to_none=True); running=0.0; seen=0; epoch_start=time.perf_counter()
        for batch_i,batch in enumerate(train_loader,start=1):
            y=batch['labels'].to(DEVICE); logits=final_token_logits(model,batch)
            loss=F.cross_entropy(logits.float(),y,weight=class_weights)
            (loss/GRADIENT_ACCUMULATION_STEPS).backward()
            running+=float(loss.item())*len(y); seen+=len(y)
            if batch_i%GRADIENT_ACCUMULATION_STEPS==0 or batch_i==len(train_loader):
                torch.nn.utils.clip_grad_norm_(trainable,GRADIENT_CLIP_NORM)
                optimiser.step(); scheduler.step(); optimiser.zero_grad(set_to_none=True)
        vr=evaluate_classifier(model,val_loader)
        row={'scenario':scenario,'epoch':epoch,'training_loss':running/seen,'validation_loss':vr['loss'],
             'validation_pr_auc':float(vr['pr_auc']),'epoch_seconds':time.perf_counter()-epoch_start}
        history.append(row); print(row)
        improved=(vr['pr_auc']>best_pr) or (np.isclose(vr['pr_auc'],best_pr) and vr['loss']<best_loss)
        if improved:
            best_pr=float(vr['pr_auc']); best_loss=float(vr['loss']); best_epoch=epoch; no_improve=0
            if best_dir.exists():
                import shutil; shutil.rmtree(best_dir)
            model.save_pretrained(best_dir); processor.save_pretrained(best_dir)
            head_state={n:t.detach().cpu() for n,t in model.state_dict().items() if 'lm_head' in n}
            assert head_state; torch.save(head_state,model_dir/'binary_classification_head.pt')
        else:
            no_improve+=1
        if no_improve>=EARLY_STOPPING_PATIENCE:
            print(f'{scenario}: early stopping after epoch {epoch}; best epoch {best_epoch}')
            break

    training_seconds=time.perf_counter()-started
    peak_gib=torch.cuda.max_memory_allocated()/1024**3
    pd.DataFrame(history).to_csv(result_dir/'training_history.csv',index=False)
    del model,optimiser,scheduler,trainable; gc.collect(); torch.cuda.empty_cache()

    base_model,saved_processor=FastVisionModel.from_pretrained(
        MODEL_ID,load_in_4bit=False,load_in_16bit=True,max_seq_length=MAX_SEQ_LENGTH,use_gradient_checkpointing=False)
    saved_tokenizer=getattr(saved_processor,'tokenizer',saved_processor); saved_tokenizer.padding_side='right'
    if saved_tokenizer.pad_token_id is None: saved_tokenizer.pad_token=saved_tokenizer.eos_token
    old_head=base_model.get_output_embeddings()
    head=FP32ClassificationHead(old_head.in_features,2,bias=False,device=old_head.weight.device,dtype=torch.float32)
    base_model.set_output_embeddings(head); base_model.config.num_labels=2; base_model.config.pad_token_id=saved_tokenizer.pad_token_id
    selected=PeftModel.from_pretrained(base_model,best_dir,is_trainable=False); selected.config.use_cache=False; selected.eval()
    collate2=make_collate(saved_tokenizer)

    test_loader=DataLoader(AssessmentTextDataset(test_frame),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate2,num_workers=0)
    forget_loader=DataLoader(AssessmentTextDataset(forget_frame),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate2,num_workers=0)
    val_loader2=DataLoader(AssessmentTextDataset(val_frame),batch_size=BATCH_SIZE,shuffle=False,collate_fn=collate2,num_workers=0)
    val_result=evaluate_classifier(selected,val_loader2); test_result=evaluate_classifier(selected,test_loader); forget_result=evaluate_classifier(selected,forget_loader)
    metrics=compute_metrics(test_result['labels'],test_result['probabilities'],FROZEN_THRESHOLD)

    pd.DataFrame([metrics]).to_csv(result_dir/'retained_test_metrics.csv',index=False)
    pd.DataFrame({'assessment_id':test_result['assessment_ids'],'label':test_result['labels'],'probability_class_1':test_result['probabilities']}).to_csv(result_dir/'retained_test_probabilities.csv',index=False)
    pd.DataFrame({'assessment_id':forget_result['assessment_ids'],'label':forget_result['labels'],'probability_class_1':forget_result['probabilities']}).to_csv(result_dir/'forget_set_probabilities.csv',index=False)

    summary={'status':'complete','scenario':scenario,'training_forget_rows':len(forget_frame),'retained_training_rows':len(train_frame),
             'retained_validation_rows':len(val_frame),'retained_test_rows':len(test_frame),'best_epoch':int(best_epoch),
             'best_validation_pr_auc':best_pr,'best_validation_loss':best_loss,'selected_checkpoint_validation_pr_auc':float(val_result['pr_auc']),
             'frozen_threshold':FROZEN_THRESHOLD,'training_seconds':training_seconds,'peak_gpu_memory_gib':peak_gib,
             **{f'test_{k}':v for k,v in metrics.items()},'adapter_path':str(best_dir)}
    complete.write_text(json.dumps(summary,indent=2),encoding='utf-8')
    del selected,base_model,saved_processor; gc.collect(); torch.cuda.empty_cache()
    print(scenario,'COMPLETE')
    return summary

## 12. Pre-flight checks — run before spending GPU time

In [15]:
preflight=pd.DataFrame([
    {'check':'CUDA available','pass':torch.cuda.is_available()},
    {'check':'60,000 assessments','pass':len(assessments)==60000},
    {'check':'42,024 training rows','pass':len(split_frames['train'])==42024},
    {'check':'same 18 features','pass':FEATURES==EXPECTED_FEATURES},
    {'check':'membership file exists','pass':MEMBERSHIP_PATH.is_file()},
    {'check':'forget counts exact','pass':scenario_size_table['training_forget_rows'].tolist()==[426,1992,4314,4148,6262]},
    {'check':'final LR = 1e-5','pass':LEARNING_RATE==1e-5},
    {'check':'LoRA rank = 16','pass':int(baseline_config['training']['LoRA rank'])==16},
    {'check':'threshold loaded','pass':0<FROZEN_THRESHOLD<1},
])
preflight['status']=preflight['pass'].map({True:'PASS',False:'FAIL'})
display(preflight[['check','status']])
assert preflight['pass'].all(), 'Pre-flight checks failed — do not start training.'
print('ALL PRE-FLIGHT CHECKS PASSED')

,check,status
0,CUDA available,PASS
1,"60,000 assessments",PASS
2,"42,024 training rows",PASS
3,same 18 features,PASS
4,membership file exists,PASS
5,forget counts exact,PASS
6,final LR = 1e-5,PASS
7,LoRA rank = 16,PASS
8,threshold loaded,PASS


ALL PRE-FLIGHT CHECKS PASSED


## 13. Run Full Retraining

**Safer first run:** temporarily set `SCENARIOS_TO_RUN = ['recipient_withdrawal']`. Confirm that the adapter and results save correctly. Then change it to `SCENARIOS` for the remaining references.

In [ ]:
# Cheap smoke-test option:
# SCENARIOS_TO_RUN = ['recipient_withdrawal']
SCENARIOS_TO_RUN = SCENARIOS

all_summaries=[]
for scenario in SCENARIOS_TO_RUN:
    print('\n' + '='*80)
    print('FULL RETRAINING:', scenario)
    print('='*80)
    all_summaries.append(train_full_retraining_scenario(scenario,scenario_sets[scenario]))

display(pd.DataFrame(all_summaries))


FULL RETRAINING: recipient_withdrawal
==((====))==  Unsloth 2026.8.22: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.25 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:1377: UserWarning: Model has `tie_word_embeddings=True` and a tied layer is part of the adapter, but `ensure_weight_tying` is not set to True. This can lead to complications, for example when merging the adapter or converting your model to formats other than safetensors. Check the discussion here: https://github.com/huggingface/peft/issues/2777
  warnings.warn(msg)
[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3_5VisionModel; please override in the subclass.. Falling back to pre-forward hook.


Unsloth: Allowing gradients for `base_model.model.lm_head` since it's in `modules_to_save`.
{'scenario': 'recipient_withdrawal', 'epoch': 1, 'training_loss': 0.6412060682875553, 'validation_loss': 0.48777818649016513, 'validation_pr_auc': 0.10910899751058013, 'epoch_seconds': 1082.6542104575783}


Unsloth: Restored added_tokens_decoder metadata in /workspace/qwen35_classifier_runpod/full_retraining/20260829T185449Z/models/recipient_withdrawal/best_adapter/tokenizer_config.json.


## 14. Combine completed scenario results

In [ ]:
completed=[]
for scenario in SCENARIOS:
    p=RESULT_ROOT/scenario/'COMPLETE.json'
    if p.is_file(): completed.append(json.loads(p.read_text(encoding='utf-8')))
completed_df=pd.DataFrame(completed)
if len(completed_df):
    completed_df.to_csv(RESULT_ROOT/'full_retraining_summary.csv',index=False)
    cols=['scenario','training_forget_rows','retained_training_rows','best_epoch','best_validation_pr_auc',
          'test_pr_auc','test_balanced_accuracy','test_binary_cross_entropy','test_f1','test_auroc','training_seconds']
    display(completed_df[[c for c in cols if c in completed_df.columns]])
print(f'Completed {len(completed_df)} / {len(SCENARIOS)} scenarios')

## 15. Final verification and manifest

In [ ]:
rows=[]
for scenario in SCENARIOS:
    rd=RESULT_ROOT/scenario; mdp=MODEL_ROOT/scenario
    required={'complete':rd/'COMPLETE.json','history':rd/'training_history.csv','test metrics':rd/'retained_test_metrics.csv',
              'test probabilities':rd/'retained_test_probabilities.csv','forget probabilities':rd/'forget_set_probabilities.csv',
              'adapter config':mdp/'best_adapter'/'adapter_config.json','classification head':mdp/'binary_classification_head.pt'}
    for name,path in required.items(): rows.append({'scenario':scenario,'artefact':name,'exists':path.exists(),'path':str(path)})
verification=pd.DataFrame(rows); display(verification)
all_complete=bool(len(verification) and verification['exists'].all())
print('ALL FIVE REFERENCES COMPLETE:',all_complete)
manifest={'run_id':RUN_ID,'baseline_run_id':BASELINE_RUN_ID,'model_id':MODEL_ID,
          'method':'fresh pretrained base + LoRA/head retraining on retained training only',
          'scenario_order':SCENARIOS,'expected_training_forget_counts':EXPECTED_TRAINING_FORGET_COUNTS,
          'frozen_threshold':FROZEN_THRESHOLD,'output_root':str(RUN_ROOT),'all_complete':all_complete}
(RESULT_ROOT/'run_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

## 16. Outputs for the later Gradient Difference notebook

Each scenario saves the Full Retraining adapter, training history, retained-test metrics, retained-test probabilities and **forget-set probabilities**. Those are the gold-standard references for later comparison of retained utility, forgetting behaviour (including Truth Ratio / KS), and runtime.

No approximate unlearning is performed here.